### This notebook is dedicated to plot the dynamic spectrum of radio emissions observed by the Parker Solar Probe (PSP), the FIELDS Radio Frequency Spectrometer (RFS) instrument 
##### Compiled by: Mohamed Nedal, Institute of Astronomy with National Astronomical Observatory, Bulgarian Academy of Sciences, Sofia, Bulgaria 
##### Many thanks to: Bing Ma$^{1,2}$, Marc Pulupa$^3$, Dejin Wu$^1$, Kamen Kozarev$^5$, Peijin Zhang$^5$, and Jon Vandegriff$^4$ 
$^1$ Key Laboratory of Planetary Sciences, Purple Mountain Observatory, Chinese Academy of Sciences, Nanjing 210023, People’s Republic of China 

$^2$ School of Astronomy and Space Science, University of Science and Technology of China, Hefei 230026, People’s Republic of China 

$^3$ Space Sciences Laboratory, University of California, Berkeley, CA 94720-7450, USA 

$^4$ JHU Applied Physics Laboratory, 11100 Johns Hopkins Road Laurel, Maryland 20723, USA 

$^5$ Institute of Astronomy with National Astronomical Observatory, Bulgarian Academy of Sciences, Sofia, Bulgaria 

* Description of parameters can be found in [this website.](https://hpde.io/NASA/NumericalData/ParkerSolarProbe/FIELDS/RFS/Level2/HFR/PT7S.html) 
* CDF data files can be found in [this website.](http://research.ssl.berkeley.edu/data/psp/data/sci/fields/) 

In [2]:
import os
os.environ['CDF_LIB'] = '/home/peijin/cdf/cdf38_0-dist/lib'
import glob
import json
import datetime
import numpy as np
import pandas as pd
from spacepy import pycdf
from astropy.io import fits
# pd.set_option('display.max_rows', None, 'display.max_columns', None)
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
# myFmt = DateFormatter('%Y/%m/%d %H:%M')
myFmt = DateFormatter('%H:%M')
import astropy.units as u
from astropy.constants import c
from astropy.time import Time
from astropy.time import TimeDelta
from sunpy.coordinates import HeliographicStonyhurst
from sunpy.coordinates import get_body_heliographic_stonyhurst
import astrospice
import warnings
warnings.filterwarnings('ignore')

In [3]:
cdf_files = []
cdf_files.append('psp_fld_l2_rfs_lfr_20190401_v02.cdf')
cdf_files.append('psp_fld_l2_rfs_hfr_20190401_v02.cdf')

In [4]:
cdf_lfr = pycdf.CDF(cdf_files[0])
cdf_hfr = pycdf.CDF(cdf_files[1])

### Define the parameters for the Low Frequncy Receiver (LFR): 10.5 kHz $-$ 1.7 MHz 

In [5]:
tmin_lfr = cdf_lfr['epoch_lfr'].meta['SCALEMIN']
tmax_lfr = cdf_lfr['epoch_lfr'].meta['SCALEMAX']

# convert pixels values to dB, # z-axis 
arr_lfr = np.array(cdf_lfr.get('psp_fld_l2_rfs_lfr_auto_averages_ch0_V1V2'))
# the min power scaled power spectral density (PSD) of 1e-16 is used as a threshold according to Pulupa et al. 2020, https://doi.org/10.3847/1538-4365/ab5dc0 
# more info: https://en.wikipedia.org/wiki/Decibel 
Lp_lfr = 10 * np.log10(arr_lfr/10**-16)
# x-axis 
tm_lfr = np.array(cdf_lfr.get('epoch_lfr'))
# y-axis 
freq_lfr = np.array(cdf_lfr.get('frequency_lfr_auto_averages_ch0_V1V2'))

### Define the parameters for the High Frequncy Receiver (HFR): 1.3 $-$ 19.2 MHz 

In [6]:
tmin_hfr = cdf_hfr['epoch_hfr'].meta['SCALEMIN']
tmax_hfr = cdf_hfr['epoch_hfr'].meta['SCALEMAX']

# convert pixels values to dB, # z-axis 
arr_hfr = np.array(cdf_hfr.get('psp_fld_l2_rfs_hfr_auto_averages_ch0_V1V2'))
# the min power scaled power spectral density (PSD) of 1e-16 is used as a threshold according to Pulupa et al. 2020, https://doi.org/10.3847/1538-4365/ab5dc0 
# more info: https://en.wikipedia.org/wiki/Decibel 
Lp_hfr = 10 * np.log10(arr_hfr/10**-16)
# x-axis 
tm_hfr = np.array(cdf_hfr.get('epoch_hfr'))
# y-axis 
freq_hfr = np.array(cdf_hfr.get('frequency_hfr_auto_averages_ch0_V1V2'))

In [7]:
tmin_hfr, tmin_lfr

(datetime.datetime(2019, 4, 1, 0, 0), datetime.datetime(2019, 4, 1, 0, 0))

### Plot the Dynamic Spectra of Radio Emissions 

In [ ]:
fig = plt.subplots(2, figsize=(20,12), dpi=70)

plt.subplot(2, 1, 1)
plt.pcolormesh(tm_hfr, freq_hfr[1]/10**6, Lp_hfr.T, vmin=0, vmax=50, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('PSP, RFS instrument, HFR band: {} $-$ {}'.format(tmin_hfr, tmax_hfr))
plt.gca().xaxis.set_major_formatter(myFmt)

plt.subplot(2, 1, 2)
plt.pcolormesh(tm_lfr, freq_lfr[1]/10**6, Lp_lfr.T, vmin=0, vmax=50, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('PSP, RFS instrument, LFR band: {} $-$ {}'.format(tmin_lfr, tmax_lfr))
plt.gca().xaxis.set_major_formatter(myFmt)

plt.tight_layout()
plt.savefig('spectrum_{}.png'.format(str(tmin_hfr)[:10]), format='png')
plt.show()

In [ ]:
fig = plt.subplots(2, figsize=(20,12), dpi=70)

plt.subplot(2, 1, 1)
plt.pcolormesh(tm_hfr, freq_hfr[1]/10**6, Lp_hfr.T, vmin=0, vmax=50, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.yscale('log')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('PSP, RFS instrument, HFR band: {} $-$ {}'.format(tmin_hfr, tmax_hfr))
plt.gca().xaxis.set_major_formatter(myFmt)

plt.subplot(2, 1, 2)
plt.pcolormesh(tm_lfr, freq_lfr[1]/10**6, Lp_lfr.T, vmin=0, vmax=50, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.yscale('log')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('PSP, RFS instrument, LFR band: {} $-$ {}'.format(tmin_lfr, tmax_lfr))
plt.gca().xaxis.set_major_formatter(myFmt)

plt.tight_layout()
plt.savefig('spectrum_logscale_{}.png'.format(str(tmin_hfr)[:10]), format='png')
plt.show()

### Combine both bands together in a single spectrum 

In [10]:
df_lfr = pd.DataFrame(Lp_lfr.T)
df_lfr.insert(loc=0, column='frequency', value=freq_lfr[1]/10**6)
df_lfr.set_index(['frequency'], inplace=True)

In [11]:
df_hfr = pd.DataFrame(Lp_hfr.T)
df_hfr.insert(loc=0, column='frequency', value=freq_hfr[1]/10**6)
df_hfr.set_index(['frequency'], inplace=True)

In [12]:
# concat the 2 arrays of both bands 
# drop the overlapped rows, take only the first row of the duplicated group 
df_psp = pd.concat([df_lfr, df_hfr])
df_psp = df_psp.sort_index(axis=0)
df_psp = df_psp[~df_psp.index.duplicated(keep='first')]

# drop the last column (timestep) because the LFR doesn't have it 
df_psp = df_psp.iloc[:,:-1]
tm_hfr = tm_hfr[:-1]

### Plot the Full Dynamic Spectrum 

In [ ]:
plt.figure(figsize=(10,6), dpi=100)
plt.pcolormesh(tm_lfr, df_psp.index, df_psp.values, vmin=0, vmax=25, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('PSP, RFS instrument: {} $-$ {}'.format(str(tm_hfr[0].replace(microsecond=0)), str(tm_hfr[-1].replace(microsecond=0))))
plt.gca().xaxis.set_major_formatter(myFmt)
plt.tight_layout()
plt.savefig('full_spectrum_{}.png'.format(str(tm_hfr[0].replace(microsecond=0))[:10]), format='png')
plt.show()

### Clean the data by subtracting the average frequency from each channel over all timesteps 

In [14]:
# calculate the mean intensity in each row (freq channel) 
df_psp_mean = df_psp.mean(axis=1)
# subtract that mean value from each corresponding row 
df_psp = df_psp.sub(df_psp_mean, axis=0)

### Plot the Dynamic Spectrum After Subtracting the Mean 

In [ ]:
plt.figure(figsize=(10,6), dpi=100)
plt.pcolormesh(tm_lfr, df_psp.index, df_psp.values, vmin=0, vmax=25, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('PSP, RFS instrument: {} $-$ {}\nAfter subtracting the mean'.format(str(tm_hfr[0].replace(microsecond=0)), str(tm_hfr[-1].replace(microsecond=0))))
plt.gca().xaxis.set_major_formatter(myFmt)
plt.tight_layout()
plt.savefig('full_spectrum_subMean_{}.png'.format(str(tm_hfr[0].replace(microsecond=0))[:10]), format='png')
plt.show()

In [ ]:
plt.figure(figsize=(10,6), dpi=100)
plt.pcolormesh(tm_lfr, df_psp.index, df_psp.values, vmin=0, vmax=20, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.yscale('log')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('PSP, RFS instrument: {} $-$ {}\nAfter subtracting the mean'.format(str(tm_hfr[0].replace(microsecond=0)), str(tm_hfr[-1].replace(microsecond=0))))
plt.gca().xaxis.set_major_formatter(myFmt)
plt.tight_layout()
plt.savefig('full_spectrum_subMean_logscale_{}.png'.format(str(tm_hfr[0].replace(microsecond=0))[:10]), format='png')
plt.show()

### WIND/WAVES Data 

In [17]:
wind = pycdf.CDF('/home/mnedal/git/psp/wi_h1s_wav_20190401000030_20190401235930.cdf')

In [21]:
wind_time = np.array(wind.get('Epoch'))
freq_RAD1 = np.array(wind.get('Frequency_RAD1'))
freq_RAD2 = np.array(wind.get('Frequency_RAD2'))
freq_TNR = np.array(wind.get('Frequency_TNR'))
spect_RAD1 = np.array(wind.get('E_VOLTAGE_RAD1'))
spect_RAD2 = np.array(wind.get('E_VOLTAGE_RAD2'))
spect_TNR = np.array(wind.get('E_VOLTAGE_TNR'))

In [ ]:
plt.figure(figsize=(10,3), dpi=100)
plt.pcolormesh(wind_time, freq_RAD1, spect_RAD1.T, vmin=-5, vmax=20, cmap='jet')
plt.colorbar(label='V', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (kHz)')
plt.gca().xaxis.set_major_formatter(myFmt)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,3), dpi=100)
plt.pcolormesh(wind_time, freq_RAD2, spect_RAD2.T, vmin=-5, vmax=20, cmap='jet')
plt.colorbar(label='V', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (kHz)')
plt.gca().xaxis.set_major_formatter(myFmt)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,3), dpi=100)
plt.pcolormesh(wind_time, freq_TNR, spect_TNR.T, vmin=-5, vmax=20, cmap='jet')
plt.colorbar(label='V', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (kHz)')
# plt.yscale('log')
plt.gca().xaxis.set_major_formatter(myFmt)
plt.tight_layout()
plt.show()

### Convert the Voltage into dB 

In [25]:
min_threshold = 1
Lp_RAD1 = 10 * np.log10(spect_RAD1/min_threshold)
Lp_RAD2 = 10 * np.log10(spect_RAD2/min_threshold)
Lp_TNR = 10 * np.log10(spect_TNR/min_threshold)

In [ ]:
plt.figure(figsize=(10,7), dpi=100)

plt.subplot(3,1,1)
plt.pcolormesh(wind_time, freq_RAD2, Lp_RAD2.T, vmin=0, vmax=15, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (kHz)')
#plt.yscale('log')
plt.gca().xaxis.set_major_formatter(myFmt)

plt.subplot(3,1,2)
plt.pcolormesh(wind_time, freq_RAD1, Lp_RAD1.T, vmin=0, vmax=15, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (kHz)')
#plt.yscale('log')
plt.gca().xaxis.set_major_formatter(myFmt)

plt.subplot(3,1,3)
plt.pcolormesh(wind_time, freq_TNR, Lp_TNR.T, vmin=0, vmax=15, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (kHz)')
#plt.yscale('log')
plt.gca().xaxis.set_major_formatter(myFmt)

plt.tight_layout()
plt.show()

In [28]:
# combine the 3 plots 
df_RAD1 = pd.DataFrame(Lp_RAD1.T)
df_RAD1.insert(loc=0, column='frequency', value=freq_RAD1)
df_RAD1.set_index(['frequency'], inplace=True)

df_RAD2 = pd.DataFrame(Lp_RAD2.T)
df_RAD2.insert(loc=0, column='frequency', value=freq_RAD2)
df_RAD2.set_index(['frequency'], inplace=True)

df_TNR = pd.DataFrame(Lp_TNR.T)
df_TNR.insert(loc=0, column='frequency', value=freq_TNR)
df_TNR.set_index(['frequency'], inplace=True)

# concat the 3 arrays 
# drop the overlapped rows, take only the first row of the duplicated group 
df_wind = pd.concat([df_RAD2, df_RAD1, df_TNR])
df_wind = df_wind.sort_index(axis=0)
df_wind = df_wind[~df_wind.index.duplicated(keep='first')]

In [41]:
print('Wind frequency ranges:\n=======================')
print('RAD1: {:.2f} - {:.2f} kHz'.format(freq_RAD1[0], freq_RAD1[-1]))
print('RAD2: {:.2f} - {:.2f} kHz'.format(freq_RAD2[0], freq_RAD2[-1]))
print('TNR: {:.2f} - {:.2f} kHz'.format(freq_TNR[0], freq_TNR[-1]))

Wind frequency ranges:
RAD1: 20.00 - 1040.00 kHz
RAD2: 1075.00 - 13825.00 kHz
TNR: 4.00 - 245.15 kHz


In [ ]:
# plot the combined spectrum 
plt.figure(figsize=(10,6), dpi=100)
plt.pcolormesh(wind_time, df_wind.index, df_wind.values, vmin=0, vmax=20, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (kHz)')
plt.gca().xaxis.set_major_formatter(myFmt)
plt.tight_layout()
plt.savefig('wind_spectrum_{}.png'.format(str(wind_time[0])[:10]), format='png')
plt.show()

In [ ]:
# plot the combined spectrum 
plt.figure(figsize=(10,6), dpi=100)
plt.pcolormesh(wind_time, df_wind.index, df_wind.values, vmin=0, vmax=20, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (kHz)')
plt.yscale('log')
plt.gca().xaxis.set_major_formatter(myFmt)
plt.tight_layout()
plt.savefig('wind_logSpectrum_{}.png'.format(str(wind_time[0])[:10]), format='png')
plt.show()

### Subtract the Mean 

In [ ]:
# replace -inf with NaNs 
df_wind = df_wind.replace(-np.inf, np.nan)

In [ ]:
# calculate the mean intensity in each row (freq channel) 
df_wind_mean = np.nanmean(df_wind.values, axis=1)
# subtract that mean value from each corresponding row 
df_wind = df_wind.sub(df_wind_mean, axis=0)

In [ ]:
# plot the combined spectrum 
plt.figure(figsize=(10,6), dpi=100)
plt.pcolormesh(wind_time, df_wind.index, df_wind.values, vmin=0, vmax=20, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (kHz)')
plt.gca().xaxis.set_major_formatter(myFmt)
plt.tight_layout()
plt.savefig('wind_spectrum_subMean_{}.png'.format(str(wind_time[0])[:10]), format='png')
plt.show()

In [ ]:
# plot the combined spectrum 
plt.figure(figsize=(18,6), dpi=100)

plt.subplot(1, 2, 1)
freq_MHz = df_wind.index.values/1000
plt.pcolormesh(wind_time, freq_MHz, df_wind.values, vmin=0, vmax=20, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('WIND/WAVE')
plt.gca().xaxis.set_major_formatter(myFmt)

plt.subplot(1, 2, 2)
plt.pcolormesh(tm_lfr, df_psp.index, df_psp.values, vmin=0, vmax=20, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('PSP/RFS')
plt.gca().xaxis.set_major_formatter(myFmt)

plt.tight_layout()
plt.savefig('wind_psp_spectra_subMean_{}.png'.format(str(wind_time[0])[:10]), format='png')
plt.show()

### STEREO/SWAVES 

In [42]:
cdf_stereo = pycdf.CDF('/home/mnedal/git/psp/stereo_level2s_swaves_20190401000030_20190401235930.cdf')

In [43]:
time_ste = np.array(cdf_stereo.get('Epoch'))
freq_ste = np.array(cdf_stereo.get('frequency'))
data_ste_A = np.array(cdf_stereo.get('avg_intens_ahead'))

In [44]:
freq_ste[0], freq_ste[-1]

(2.609999895095825, 16025.0)

In [45]:
cdf_stereo['frequency'].meta

<zAttrList:
CATDESC: Frequency [CDF_CHAR]
DEPEND_0: Epoch [CDF_CHAR]
DIM_SIZES: 367 [CDF_INT4]
DISPLAY_TYPE: time_series [CDF_CHAR]
FIELDNAM: Frequency [CDF_CHAR]
FILLVAL: -1e+31 [CDF_FLOAT]
FORMAT: f9.3 [CDF_CHAR]
LABLAXIS: Frequency [CDF_CHAR]
SCALETYP: log [CDF_CHAR]
UNITS: kHz [CDF_CHAR]
VALIDMAX: 16025.0 [CDF_FLOAT]
VALIDMIN: 2.6 [CDF_FLOAT]
VAR_TYPE: support_data [CDF_CHAR]
>

In [ ]:
plt.figure(figsize=(10,5), dpi=100)
plt.pcolormesh(time_ste, freq_ste/1000, data_ste_A.T, vmin=0, vmax=20, cmap='jet')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.colorbar(label='dB', pad=0.01)
plt.gca().xaxis.set_major_formatter(myFmt)
plt.tight_layout()
plt.show()

### Subtract the Mean 

In [ ]:
# calculate the mean intensity in each row (freq channel) 
df_ste_A = pd.DataFrame(data_ste_A.T)
df_ste_mean = df_ste_A.mean(axis=1)
# subtract that mean value from each corresponding row 
data_ste_A = df_ste_A.sub(df_ste_mean, axis=0)

In [ ]:
plt.figure(figsize=(10,5), dpi=100)
plt.pcolormesh(time_ste, freq_ste/1000, data_ste_A, vmin=0, vmax=20, cmap='jet')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.colorbar(label='dB', pad=0.01)
plt.gca().xaxis.set_major_formatter(myFmt)
plt.tight_layout()
plt.show()

### LOFAR Data 

In [47]:
def get_data_from_fits(path):
    with fits.open(path) as hdu:
        data = hdu[0].data.astype(np.float64)
    return data

directory = '/data/LOFAR/events/LBA/'
paths_fits = glob.glob(directory + 'LOFAR_20190401_*.fits')
paths_json = glob.glob(directory + 'LOFAR_20190401_*.json')
paths_fits.sort()
paths_json.sort()

# Returns JSON object as a dictionary 
data = []
for file in paths_json:
    f = open(file)
    data.append(json.load(f))
    f.close()

df_all = []
for iterator in range((len(paths_fits))):
    # read the file 
    fits_data = fits.open(paths_fits[iterator])[0]
    arr = fits_data.data.T
    # get time info 
    start_obs = data[iterator]['time_range'][0]
    end_obs = data[iterator]['time_range'][1]
    # get freq info 
    first_freq = data[iterator]['freq_range'][0]
    last_freq = data[iterator]['freq_range'][1]
    num_steps_freq = data[iterator]['n_freq']
    freq_step = last_freq/num_steps_freq
    # define time and freq axes 
    freq_axis = np.arange(first_freq, last_freq, freq_step-0.0494)
    time_axis = pd.date_range(start=start_obs, end=end_obs, periods=fits_data.header['NAXIS2'])
    # clean the data by subtracting the mean 
    df = pd.DataFrame(arr)
    # calculate the mean intensity in each row (freq channel) 
    df_mean = df.mean(axis=1)
    # subtract that mean value from each corresponding row 
    df_submean = df.sub(df_mean, axis=0)
    # transpose the table 
    df_submean = df_submean.T
    # insert the datetimes as index 
    df_submean.insert(loc=0, column='DateTime', value=time_axis)
    df_submean.set_index(['DateTime'], inplace=True)
    # store the cleaned tables 
    df_all.append(df_submean)

# Concat. the list based on time index 
df_all = pd.concat(df_all, axis=0)
# convert to string to remove the milliseconds 
df_all.index = df_all.index.strftime('%Y-%m-%d %H:%M:%S')
# convert it back to datetime object 
df_all.index = [datetime.datetime.strptime(tm, '%Y-%m-%d %H:%M:%S') for tm in df_all.index]

correct_time_axis = pd.date_range(start=df_all.index[0], end=df_all.index[-1], freq='S')
# Drop rows with duplicate index values 
df_all = df_all.loc[~df_all.index.duplicated(), :]
new_df_all = df_all.reindex(correct_time_axis, fill_value=np.nan)

In [48]:
# STEREO 
time_ste[1] - time_ste[0]

datetime.timedelta(seconds=60)

In [49]:
# WIND 
wind_time[1] - wind_time[0]

datetime.timedelta(seconds=60)

In [50]:
# PSP 
tm_lfr[1] - tm_lfr[0]

datetime.timedelta(seconds=6, microseconds=990479)

In [51]:
# LOFAR 
new_df_all.index[1] - new_df_all.index[0]

Timedelta('0 days 00:00:01')

In [ ]:
plt.figure(figsize=(18,7), dpi=100)
plt.suptitle('2019/04/01 Radio Flux Intensity', fontsize=14, y=1.05)
plt.subplot(2, 2, 1)
freq_ste_A = freq_ste/1000
plt.pcolormesh(time_ste, freq_ste_A, data_ste_A, vmin=0, vmax=10, cmap='jet')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('STEREO-A/SWAVES, frequency range: {:.2f} - {:.2f} kHz'.format(freq_ste_A[0]*1000, freq_ste_A[-1]*1000))
plt.colorbar(label='dB', pad=0.01)
plt.gca().xaxis.set_major_formatter(myFmt)

plt.subplot(2, 2, 2)
freq_MHz = df_wind.index.values/1000
plt.pcolormesh(wind_time, freq_MHz, df_wind.values, vmin=0, vmax=20, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('WIND/WAVE, frequency range: {:.2f} - {:.2f} kHz'.format(df_wind.index[0], df_wind.index[-1]))
plt.gca().xaxis.set_major_formatter(myFmt)

plt.subplot(2, 2, 3)
plt.pcolormesh(tm_lfr, df_psp.index, df_psp.values, vmin=0, vmax=20, cmap='jet')
plt.colorbar(label='dB', pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('PSP/RFS, frequency range: {:.2f} - {:.2f} kHz'.format(df_psp.index[0]*1000, df_psp.index[-1]*1000))
plt.gca().xaxis.set_major_formatter(myFmt)

plt.subplot(2, 2, 4)
plt.pcolormesh(new_df_all.index, freq_axis, new_df_all.values.T, vmin=0, vmax=3, cmap='jet')
plt.colorbar(pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.title('LOFAR/LBA (Outer), frequency range: {:.2f} - {:.2f} kHz'.format(freq_axis[0]*1000, freq_axis[-1]*1000))
plt.gca().xaxis.set_major_formatter(myFmt)

plt.tight_layout()
plt.savefig('/home/mnedal/git/psp/4_comparison.png', format='png', dpi=300)
plt.show()

### Calculate Relative Light Travel Time for the PSP and the Earth 

In [ ]:
kernels = astrospice.registry.get_kernels('psp', 'predict')
psp_kernel = kernels[0]
coverage = psp_kernel.coverage('SOLAR PROBE PLUS')

In [ ]:
start_obstime = Time('2019-04-01T00:00:00.000')
end_obstime = Time('2019-04-01T23:51:51.000')

dt = TimeDelta(1 * u.day)
times = Time(np.arange(start_obstime, end_obstime, dt))

In [ ]:
coords = astrospice.generate_coords('SOLAR PROBE PLUS', times)

In [ ]:
new_frame = HeliographicStonyhurst()
coords = coords.transform_to(new_frame)
obj_coord = get_body_heliographic_stonyhurst('Earth', time=start_obstime)

# convert units to AU 
psp_distAU = np.array(coords.radius.to(u.AU))[0]
earth_distAU = np.array(obj_coord.radius).reshape(-1)[0]

# convert units to m 
psp_dist = np.array(coords.radius.to(u.m))[0]
earth_dist = np.array(obj_coord.radius.to(u.m)).reshape(-1)[0]

In [ ]:
print('PSP distance from the Sun:\n{:.2f} AU, or {:.2f} km.\n'.format(psp_distAU, psp_dist/10**3))
print('Earth distance from the Sun:\n{:.2f} AU, or {:.2f} km.'.format(earth_distAU, earth_dist/10**3))

In [ ]:
# get the time (seconds) for EM wave to travel from the Sun to the target (PSP, Earth) 
t_psp = psp_dist/np.array(c).reshape(-1)[0]
t_earth = earth_dist/np.array(c).reshape(-1)[0]

In [ ]:
print('Time travel for radio waves\n----------------------------')
print('For PSP: {:.2f} light seconds, or {:.2f} light minutes.'.format(t_psp, t_psp/60))
print('For Earth: {:.2f} light seconds, or {:.2f} light minutes.'.format(t_earth, t_earth/60))

In [ ]:
# set the columns names as the timestamps 
df_psp.columns = tm_lfr

In [ ]:
# transpose to set the timestamps as the index column 
df_psp = df_psp.T

In [ ]:
df_psp.head(3)

In [ ]:
# plot PSP with LOFAR -- before the shifting 
plt.figure(figsize=(10,5), dpi=300)
plt.title('2019/04/01 Radio Flux Intensity')
plt.pcolormesh(df_psp.T.columns, df_psp.T.index, df_psp.T.values, vmin=0, vmax=10, cmap='jet')
plt.pcolormesh(new_df_all.index, freq_axis, new_df_all.values.T, vmin=0, vmax=3, cmap='jet')
plt.colorbar(pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.gca().xaxis.set_major_formatter(myFmt)
plt.tight_layout()
plt.show()

In [ ]:
# take LOFAR as reference and shift PSP 
df_psp = df_psp.shift(round(t_psp), axis=0)

In [ ]:
# plot PSP with LOFAR -- after the shifting 
#plt.figure(figsize=(10,5), dpi=300)
plt.figure(figsize=(17,7))
plt.title('2019/04/01 Radio Flux Intensity, shifted PSP')
plt.pcolormesh(df_psp.T.columns, df_psp.T.index, df_psp.T.values, vmin=0, vmax=10, cmap='jet')
plt.pcolormesh(new_df_all.index, freq_axis, new_df_all.values.T, vmin=0, vmax=3, cmap='jet')
plt.colorbar(pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.gca().xaxis.set_major_formatter(myFmt)
plt.tight_layout()
plt.show()

In [ ]:
def nearest(items, pivot):
    ''' This function will return the datetime in items which is the closest to the date pivot ''' 
    return min(items, key=lambda x: abs(x - pivot))

In [ ]:
# take a snapshot 
hh_left = 14
mm_left = 0
ss_left = 0
xleft = datetime.datetime(2019, 4, 1, hh_left, mm_left, ss_left, 0)

hh_right = 15
mm_right = 25
ss_right = 0
xright = datetime.datetime(2019, 4, 1, hh_right, mm_right, ss_right, 0)

times_psp = np.array([pd.to_datetime(df_psp.T.columns[i]) for i in range(len(df_psp.T.columns))])
times_lofar = np.array([pd.to_datetime(new_df_all.index[i]) for i in range(len(new_df_all.index))])

xnear_left = nearest(times_psp, xleft)
xnear_right = nearest(times_psp, xright)

#plt.figure(figsize=(8,5), dpi=300)
plt.figure(figsize=(17,7))
plt.title('2019/04/01 Radio Flux Intensity, shifted PSP data')
plt.pcolormesh(df_psp.T.columns, df_psp.T.index, df_psp.T.values, vmin=0, vmax=10, cmap='jet')
plt.pcolormesh(new_df_all.index, freq_axis, new_df_all.values.T, vmin=0, vmax=3, cmap='jet')
plt.colorbar(pad=0.01)
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.gca().xaxis.set_major_formatter(myFmt)
plt.xlim(left=xleft, right=xright)
plt.tight_layout()
plt.show()

In [ ]:
print('Time resolution of PSP: {} seconds'.format(round((times_psp[1] - times_psp[0]).seconds)))
print('Time resolution of LOFAR: {} second'.format((times_lofar[1] - times_lofar[0]).seconds))

In [ ]:
limit1 = 200
limit2 = 500
tm_lfr_slice = tm_lfr[limit1:limit2]
df_psp_freq_slice = df_psp.index
df_psp_val_slice = df_psp.values[:,limit1:limit2]

plt.figure(figsize=(20,5))
plt.pcolormesh(tm_lfr_slice, df_psp_freq_slice, df_psp_val_slice, vmin=0, vmax=20, cmap='jet')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.gca().xaxis.set_major_formatter(myFmt)
plt.show()

In [ ]:
hh = 0
mm = 44
ss = 30
xc = datetime.datetime(2019, 4, 1, hh, mm, ss, 0)

def nearest(items, pivot):
    ''' This function will return the datetime in items which is the closest to the date pivot ''' 
    return min(items, key=lambda x: abs(x - pivot))


from scipy.signal import argrelextrema

plt.figure(figsize=(20,5))
plt.pcolor(tm_lfr_slice, df_psp_freq_slice, df_psp_val_slice, vmin=0, vmax=20, cmap='jet')
locs, labels = plt.yticks()
xc_near = nearest(tm_lfr_slice, xc)
plt.vlines(x=xc_near, ymin=locs[0], ymax=locs[-1], color='white', linewidth=3, linestyle='--')
plt.xlabel('Time (UT)')
plt.ylabel('Frequency (MHz)')
plt.gca().xaxis.set_major_formatter(myFmt)
plt.show()

plt.figure(figsize=(20,3))
idx = np.where(tm_lfr_slice == xc_near)[0][0]
plt.plot(df_psp_val_slice.T[idx])
locs2, labels2 = plt.yticks()
plt.vlines(np.argmax(df_psp_val_slice.T[idx]), ymin=locs2[0], ymax=locs2[-1], color='green', linewidth=4, linestyle='--')
peaks = argrelextrema(df_psp_val_slice.T[idx], np.greater)
for i in peaks:
    plt.vlines(i, ymin=locs2[0], ymax=locs2[-1], color='red', linestyle='--')
plt.show()

# subtract avg of the first (lowest) few freq. channels from the data since there's nothing happening there ... 
# plot the sum of the freq channels vs. time ... 
# determine the "timing" of the bursts 